In [2]:
#pd.set_option('display.max_columns', None)
import pandas as pd
import os 
pd.set_option('display.max_rows', None)        # show all rows
pd.set_option('display.max_columns', None)     # show all columns
pd.set_option('display.width', None)           # auto-detect width
pd.set_option('display.max_colwidth', None) 

path_name = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/processed/df_03_27_2026_aiff_tracks_data.pkl'

df= pd.read_pickle(path_name)

df_raw= pd.read_pickle(path_name)

print(len(df))

# check if paths exist for analized songs 

print(df['Path'].apply(lambda x: os.path.exists(x)).all())


2402
True


### FILTER  = LLM + RAG

In [5]:
df.head(3)

,ID,Path,Extension,dur_seconds,dur_min,sr,bit_depth,channels,file_size,file_size_human,num_frames,ms_lufs,id_cat_lufs,mean_bpm,std_bpm,min_bpm,max_bpm,variation_percentage,dominant_bpm,bpm_consistency,bpm_consistency_cat,title,title_file,artist,artist_file,LABEL,label_file,genre,genre_file,rel_year,rel_year_file,KEY,key_file,mix_name,remixer,remix,date_purchased,rel_date,rel_date_file,key_dj,key_music,Relative_Key,Key_Up,Key_Down,Jaw_s_Mix,Mood_Shifter,comment,ms_LUFS_norm,Spectral_Bandwidth,Spectral_Flatness,HEX_shape_texture,spec_centroid_hz,centroid_color,centroid_desc,year_written_id3,bought_year,lufs_pct,audio_hash,__source_file
0,t1-12306,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0T[26]-123BPM-9A_Emin--id_t1-12306---Deep-PHONOGRAM--by--RICKWADE-functionalanger(O)-2025.aiff,.aiff,393.799206,6.56332,44100,16,2,70378472,67.12 MB,17366545,-11.53,T,123.426649,2.764799,123.046875,143.554688,2.240034,123,98.148148,D_0,Functional Anger Original Mix,ID3TAGS,Rick Wade,ID3TAGS,Phonogramme,ID3TAGS,Deep House,ID3TAGS,2025,ID3TAGS,Emin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2025-11-10,ID3TAGS,9A,Emin,9B,10A,8A,4A,12B,0LUFS--T-87%--0Emin_123BPM--id_t1-12306,76.923077,3800.914308,0.008291,#005EFF,3490.022323,#808000,"Tense, decaying",Unsupported,2026,87,622ea4787b16ecd93da93d52c6febda8c780942622aff84b45584ff2c29fc0cf,df_t6t_final.pkl
1,t1-12300,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0T[26]-123BPM-6A_Gmin--id_t1-12300---Deep-CDR(CROS--by--AARONCARLDYED-nakedfeataaron(O)-2009.aiff,.aiff,469.746667,7.829111,44100,16,2,82900338,79.06 MB,20715828,-11.03,T,123.046875,0.000000,123.046875,123.046875,0.000000,123,100.000000,D_0,Naked feat Aaron Carl Original Mix,ID3TAGS,Aaron Carl Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Gmin,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,6A,Gmin,6B,7A,5A,1A,9B,0LUFS--T-87%--0Gmin_123BPM--id_t1-12300,76.923077,4725.746003,0.032783,#113C70,5344.390429,#FFFF99,Dreamy & bright,Unsupported,2026,87,71888cd188d034fc921f1b4d8e4bc3961373e5815f4cc72c4dd1c7a49fe01f45,df_t6t_final.pkl
2,t1-12304,/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_6T[26]-126BPM-7B_Fmaj--id_t1-12304---Deep-CDR(CROS--by--DYEDSOUNDOROM-beautifulevaor(O)-2009.aiff,.aiff,386.573333,6.442889,44100,16,2,68222900,65.06 MB,17047884,-11.24,T,119.603689,15.284070,83.354335,126.048018,12.778928,126,84.905660,D_6,Beautiful Eva Original Mix,ID3TAGS,Dyed Soundorom,ID3TAGS,CDR (Crosstown Digital Rebels),ID3TAGS,Deep House,ID3TAGS,2009,ID3TAGS,Fmaj,ID3TAGS,Original_Mix,Original Mix,O,2026-01-23,2009-01-26,ID3TAGS,7B,Fmaj,7A,8B,6B,2B,4A,6LUFS--T-87%--6Fmaj_126BPM--id_t1-12304,76.923077,3003.322452,0.005540,#005EFF,2639.947662,#3C3B6E,Dubby twilight,Unsupported,2026,87,ed45bd1d38502a22a69f1adff5870f477195eb67336f4e65c71bf295e105caa4,df_t6t_final.pkl


In [3]:
# ----- FILTER EXACT MATCH -----

df = df[
    (df['bpm_consistency'].round() == 100) &
    (df['dominant_bpm'] == 126) &
    (df['genre'].str.lower().str.contains('house')) & 
    (df['key_dj'].str.contains('6A|6B'))
].copy()

df[['bpm_consistency', 'dominant_bpm', 'genre']].head()

,bpm_consistency,dominant_bpm,genre
21,100.0,126,House
23,100.0,126,House
130,100.0,126,Tech House
209,100.0,126,Jackin House
464,100.0,126,Tech House


In [4]:
# -----######-----######-----######-----######-----######
# _audio_1104_i7_GET_df_player_smartDJ
# SMART DJ ENGINE (BEAT ALIGN + ENERGY + EQ SIMULATION)
# -----######-----######-----######-----######-----######

import pygame
import random
import threading
import time
import sys
import numpy as np
import librosa
from tqdm import tqdm


# ---------- FEATURE EXTRACTION ----------
def _analyze_track(path):

    y, sr = librosa.load(path, sr=22050)

    # ---- BEAT TRACK ----
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)

    if len(beats) > 0:
        start_sec = librosa.frames_to_time(beats[0], sr=sr)
    else:
        start_sec = 0

    # ---- ENERGY ----
    rms = np.mean(librosa.feature.rms(y=y))

    # ---- FREQUENCY SPLIT ----
    S = np.abs(librosa.stft(y))

    freqs = librosa.fft_frequencies(sr=sr)

    low = S[(freqs < 200)].mean()
    mid = S[(freqs >= 200) & (freqs < 2000)].mean()
    high = S[(freqs >= 2000)].mean()

    return {
        "start_sec": start_sec,
        "energy": rms,
        "low": low,
        "mid": mid,
        "high": high
    }


# ---------- MAIN PLAYER ----------
def _audio_1104_i7_GET_df_player_smartDJ(df, fade_ms=4000):

    pygame.mixer.init()

    paths = df['Path'].dropna().tolist()

    if len(paths) == 0:
        print("❌ No valid paths found")
        return

    print("\n🔍 Analyzing tracks (one-time)...")

    meta = {}
    for p in tqdm(paths):
        try:
            meta[p] = _analyze_track(p)
        except:
            meta[p] = {"start_sec":0,"energy":0,"low":0,"mid":0,"high":0}

    print("✅ Analysis complete")

    current = {"sound": None, "channel": None, "path": None}

    lock = threading.Lock()


    # ---------- LOAD WITH OFFSET ----------
    def play_from_start(path):

        sound = pygame.mixer.Sound(path)

        # NOTE: pygame can't start at offset → workaround = fade-in illusion
        ch = sound.play(fade_ms=100)

        return sound, ch


    # ---------- SMART TRANSITION ----------
    def smart_transition(next_path):

        prev = current["path"]

        new_sound, new_channel = play_from_start(next_path)

        if prev is not None:

            prev_low = meta[prev]["low"]
            next_low = meta[next_path]["low"]

            # ---- LOW FREQ STRATEGY ----
            if next_low > prev_low:
                # delay bass → longer fade
                fade_out = int(fade_ms * 1.5)
            else:
                fade_out = fade_ms

            current["channel"].fadeout(fade_out)

        current["sound"] = new_sound
        current["channel"] = new_channel
        current["path"] = next_path


    # ---------- INIT BAR ----------
    for _ in tqdm(range(50), desc="Initializing DJ Engine"):
        time.sleep(0.01)


    print("\n🎛 SMART DJ CONTROLS")
    print("   s → start")
    print("   n → next (beat-aware)")
    print("   q → quit")


    # ---------- LOOP ----------
    while True:

        cmd = input("\n👉 [s/n/q]: ").strip().lower()

        if cmd == "s":

            path = random.choice(paths)

            with lock:
                smart_transition(path)

            print(f"\n▶️ START:\n{path}")


        elif cmd == "n":

            # ---- SMART PICK ----
            current_energy = meta[current["path"]]["energy"] if current["path"] else 0

            candidates = sorted(
                paths,
                key=lambda p: abs(meta[p]["energy"] - current_energy)
            )[:5]

            next_path = random.choice(candidates)

            with lock:
                smart_transition(next_path)

            print(f"\n⏭️ NEXT (ENERGY MATCHED):\n{next_path}")


        elif cmd == "q":

            if current["channel"]:
                current["channel"].fadeout(1000)

            print("\n🛑 STOPPED")
            break

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

_audio_1104_i7_GET_df_player_smartDJ(df)


🔍 Analyzing tracks (one-time)...


100%|█████████████████████████████████████████████████████████████████████████| 29/29 [00:31<00:00,  1.09s/it]


✅ Analysis complete


Initializing DJ Engine: 100%|█████████████████████████████████████████████████| 50/50 [00:00<00:00, 80.61it/s]



🎛 SMART DJ CONTROLS
   s → start
   n → next (beat-aware)
   q → quit



👉 [s/n/q]:  s



▶️ START:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_0V[25]-126BPM-6A_Gmin--id_t1-12120k---Deep-DISTROKID--by--GMAJORHASEO-haseosgroovef(O)-2024.aiff



👉 [s/n/q]:  n



⏭️ NEXT (ENERGY MATCHED):
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_11_zThanksgiving_SET_DET/dylu_0T[25]-126BPM-6A_Gmin--id_t1-11280p---Hous-FROLEREC--by--RICKWADE-funkyoneorigin(O)-2025.aiff
